In [1]:
import os
import glob
import pandas as pd
from functools import reduce


def load_parquet_folder(folder_name, freq='1min'):
    """
    Reads all parquet files from a folder, merges them on Timestamp,
    creates a complete time index, and returns the final DataFrame.

    Parameters
    ----------
    folder_name : str
        Path to the folder containing parquet files.
    freq : str, optional
        Frequency for the complete time index (default='1min').

    Returns
    -------
    pd.DataFrame
        Merged DataFrame with Timestamp column.
    """

    # List all parquet files
    files = glob.glob(os.path.join(folder_name, "*.parquet"))

    if len(files) == 0:
        raise ValueError(f"No parquet files found in '{folder_name}'")

    dfs = []

    for file in files:
        # Read parquet file
        df = pd.read_parquet(file)

        # Convert Timestamp column to datetime
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])

        # Remove duplicate timestamps
        df = df.drop_duplicates(subset='Timestamp')

        # Set Timestamp as index
        df = df.set_index('Timestamp')

        # Assume only one data column besides Timestamp
        value_col = df.columns[0]

        # Use filename as tag name
        tag_name = os.path.splitext(os.path.basename(file))[0]

        # Rename column to tag name
        df = df.rename(columns={value_col: tag_name})

        dfs.append(df)

    # Merge all DataFrames
    merged_df = reduce(
        lambda left, right: left.join(right, how='outer'),
        dfs
    )

    # Create complete time index
    full_index = pd.date_range(
        start=merged_df.index.min(),
        end=merged_df.index.max(),
        freq=freq
    )

    # Reindex to complete timeline
    merged_df = merged_df.reindex(full_index)

    # Restore Timestamp column
    merged_df = merged_df.reset_index()
    merged_df.rename(columns={'index': 'Timestamp'}, inplace=True)

    return merged_df

In [2]:
All_df = load_parquet_folder("All_unit_tags", freq='1min')

In [3]:
All_df = All_df.iloc[58677:]

In [4]:
All_df.head()

,Timestamp,99TI1320.DACA.PV,99TI1505A.DACA.PV,103PC1707.daca.pv,41TI4136.pv,02PI213.PV,105PI1518AN.PV,71PC9801.PV,103TIC5122N.pida.pv,105PI2608.DACA.PV,...,PTC3209.PV,99PI2360B.PV,05TI508.PV,102PI5006.DACA.PV,99PI3805.DACA.PV,99PI3705.DACA.PV,105TI2810N.PV,102PC4702.DACA.PV,41TI41136.pv,16TT110.pv
58677,2025-05-22 17:57:00,0.0,30.14,12.55,31.91,4.32,3.69,21.93,52.69,15.48,...,269.10,-0.0,0.0,8.22,742.05,738.54,47.91,24.88,138.70,32.09
58678,2025-05-22 17:58:00,0.0,30.10,12.54,31.88,4.36,3.69,21.94,53.19,15.48,...,269.26,-0.0,0.0,8.21,742.01,738.54,47.90,24.80,138.71,32.02
58679,2025-05-22 17:59:00,0.0,30.07,12.52,31.86,4.36,3.69,21.85,53.58,15.47,...,269.18,-0.0,0.0,8.21,741.97,738.54,47.90,24.88,138.80,31.96
58680,2025-05-22 18:00:00,0.0,30.04,12.50,31.82,4.36,3.68,21.79,53.87,15.47,...,268.77,-0.0,0.0,8.22,741.93,738.54,47.91,24.91,138.75,31.90
58681,2025-05-22 18:01:00,0.0,30.00,12.47,31.80,4.36,3.67,21.83,53.95,15.48,...,268.24,-0.0,0.0,8.23,741.90,738.52,47.91,24.82,138.80,31.83


In [5]:
Flare_data_df = load_parquet_folder("flare_data", freq='1min')

In [6]:
Flare_data_df = Flare_data_df.iloc[58677:]

In [7]:
import pandas as pd
import numpy as np

def clean_and_impute(
    df,
    max_missing_pct=10,
    method='spline',
    order=3,
    limit_direction='both',
    std_threshold=0.0,
    slope_std_threshold=1e-10
):
    """
    Remove columns with:
    1. Missing percentage > max_missing_pct
    2. Standard deviation <= std_threshold (constant columns)
    3. Nearly constant slope (linear trend)

    Then impute the remaining missing values.
    """

    print(f"Original shape: {df.shape}")

    # ==============================
    # Separate Timestamp column
    # ==============================
    if 'Timestamp' in df.columns:
        timestamp = df['Timestamp']
        df_num = df.drop(columns=['Timestamp']).copy()
    else:
        timestamp = None
        df_num = df.copy()

    # ==============================
    # Remove columns with excessive missing values
    # ==============================
    missing_percent = df_num.isna().mean() * 100

    cols_to_keep = missing_percent[
        missing_percent <= max_missing_pct
    ].index

    removed_missing_cols = list(
        set(df_num.columns) - set(cols_to_keep)
    )

    if removed_missing_cols:
        print(f"Removed {len(removed_missing_cols)} columns with > {max_missing_pct}% missing values.")
        print(removed_missing_cols)

    df_num = df_num[cols_to_keep]

    # ==============================
    # Remove constant columns
    # ==============================
    std_values = df_num.std()

    zero_std_cols = std_values[
        std_values <= std_threshold
    ].index.tolist()

    if zero_std_cols:
        print(f"Removed {len(zero_std_cols)} constant columns.")
        print(zero_std_cols)

    df_num = df_num.drop(columns=zero_std_cols)

    # ==============================
    # Remove constant-slope columns
    # ==============================
    slope = df_num.diff()

    slope_std = slope.std(skipna=True)

    constant_slope_cols = slope_std[
        slope_std <= slope_std_threshold
    ].index.tolist()

    if constant_slope_cols:
        print(f"Removed {len(constant_slope_cols)} constant-slope columns.")
        print(constant_slope_cols)

    df_num = df_num.drop(columns=constant_slope_cols)
    
    # ==============================
    # Remove columns containing negative values
    # ==============================
    negative_cols = df_num.columns[(df_num < 0).any()].tolist()

    if negative_cols:
         print(f"Removed {len(negative_cols)} columns containing negative values.")
         print(negative_cols)

    df_num = df_num.drop(columns=negative_cols)
    # ==============================
    # Interpolate missing values
    # ==============================
    if method == 'spline':
        df_num = df_num.interpolate(
            method='spline',
            order=order,
            limit_direction=limit_direction
        )
    else:
        df_num = df_num.interpolate(
            method=method,
            limit_direction=limit_direction
        )

    # Fill remaining NaNs
    df_num = df_num.ffill().bfill()

    # ==============================
    # Reattach Timestamp
    # ==============================
    if timestamp is not None:
        df_clean = pd.concat(
            [timestamp.reset_index(drop=True),
             df_num.reset_index(drop=True)],
            axis=1
        )
    else:
        df_clean = df_num

    # ==============================
    # Summary
    # ==============================
    print(f"Cleaned shape: {df_clean.shape}")
    print(f"Remaining missing values: {df_clean.isna().sum().sum()}")

    return df_clean

In [8]:
print("All_unit_tags")
All_clean_df=clean_and_impute(All_df)

All_unit_tags
Original shape: (523084, 377)
Removed 23 columns with > 10% missing values.
['102PC1201R.DACA.PV', '102PI1457B.DACA.PV', '102TI4405.DACA.PV', '102TI1504.DACA.PV', '102TI0701.DACA.PV', '102PI4557.DACA.PV', '102TI4602.DACA.PV', '102PI1502.DACA.PV', '102TI4301.DACA.PV', '102TI4406.DACA.PV', '700TI3936N', '102TI1515.DACA.PV', '102TI3903.DACA.PV', '102PI3906N.DACA.PV', '102PC4302R.DACA.PV', '102PC4101.DACA.PV', '101PC1105.DACA.PV', '102PI4244AN.PV', '102TI4250.DACA.PV', '102PI1507.DACA.PV', '102PI4559.DACA.PV', '102TI4351N.DACA.PV', '102PI4459.DACA.PV']
Removed 3 constant columns.
['105TI2401.DACA.PV', '105PI2803.DACA.PV', '99PI1542.DACA.PV']
Removed 2 constant-slope columns.
['PPI3239.PV', '102PY6708.PV']
Removed 63 columns containing negative values.
['02PI213.PV', '99PI4104.DACA.PV', '98PIC2601.DACA.PV', '102TI4790N.DACA.PV', '105PI2804N.PV', '103PI7009N.pv', '99PI3413.DACA.PV', '99PI1566.PV', '99PI1206.DACA.PV', '99PI3265.PV', '99PI1909.DACA.PV', '102TI4789N.DACA.PV', 'THP

In [9]:
print("Flare_data_unit")
Flare_data_clean_df=clean_and_impute(Flare_data_df, 50)

Flare_data_unit
Original shape: (523084, 5)
Cleaned shape: (523084, 5)
Remaining missing values: 0


In [10]:
Flare_data_clean_df['Flare_Flowrate'] = Flare_data_clean_df['509fi5201.daca.pv_1000'] + Flare_data_clean_df['(LRflare.pv+LREflare.pv+99fi4601.daca.pv)_1000'] - Flare_data_clean_df['(44ft2505.pv_509mw5101.daca.pv_22.414)_1000']

In [11]:
# # Decide the first normal window
# threshold = 1.5
# window = 720

# mask = Flare_data_clean_df['Flare_Flowrate'] < threshold

# # Rolling sum of True values
# rolling = mask.rolling(window=window).sum()

# # Find the first window with all 720 values below threshold
# idx = rolling[rolling == window].index

# if len(idx) > 0:
#     end_idx = idx[0]
#     start_idx = end_idx - window + 1

#     start_ts = Flare_data_clean_df.loc[start_idx, 'Timestamp']
#     end_ts = Flare_data_clean_df.loc[end_idx, 'Timestamp']

#     print(f"Found {window} consecutive samples below {threshold}")
#     print(f"Start index     : {start_idx}")
#     print(f"End index       : {end_idx}")
#     print(f"Start Timestamp : {start_ts}")
#     print(f"End Timestamp   : {end_ts}")
# else:
#     print(f"No {window} consecutive samples below {threshold} found.")

In [12]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import f

# ==========================================================
# Prepare data
# ==========================================================
X_df = All_clean_df.select_dtypes(include=np.number)

# Remove existing T2 column if present
X_df = X_df.drop(columns=['T2'], errors='ignore')

feature_names = X_df.columns
X = X_df.values

# ==========================================================
# Standardize
# ==========================================================
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# ==========================================================
# Global PCA model
# ==========================================================
pca = PCA()
scores = pca.fit_transform(Xs)

# Number of PCs to retain (95% variance)
cum_var = np.cumsum(
    pca.explained_variance_ratio_
)

A = np.argmax(cum_var >= 0.95) + 1

print(f'Number of retained PCs = {A}')

# ==========================================================
# Compute T²
# ==========================================================
Lambda = pca.explained_variance_[:A]
T = scores[:, :A]

T2 = np.sum(
    (T**2) / Lambda,
    axis=1
)

All_clean_df['T2'] = T2

# ==========================================================
# T² control limit
# ==========================================================
n = X.shape[0]
p = A

T2_limit = (1.2*
   ( p * (n - 1) * (n + 1)
    / (n * (n - p))
) * f.ppf(
    0.99,
    p,
    n - p
))

print('T² limit =', T2_limit)

# ==========================================================
# Reconstruction-Based Contribution (RBC)
# ==========================================================
P = pca.components_.T
P_A = P[:, :A]

# Reconstruction
X_rec = T @ P_A.T

# Residual matrix
E = Xs - X_rec

# RBC
RBC = E**2

RBC_df = pd.DataFrame(
    RBC,
    columns=feature_names,
    index=All_clean_df.index
)

# ==========================================================
# Alarm samples
# ==========================================================
alarm_idx = All_clean_df.loc[
    All_clean_df['T2'] > T2_limit
].index

print(
    'Number of alarm samples:',
    len(alarm_idx)
)

# ==========================================================
# Top-5 contributors for each alarm
# ==========================================================
rows = []

for idx in alarm_idx:

    contrib = (
        RBC_df.loc[idx]
        .sort_values(ascending=False)
        .head(5)
    )

    row = {
        'Timestamp':
            All_clean_df.loc[idx, 'Timestamp'],
        'T2':
            All_clean_df.loc[idx, 'T2']
    }

    for i, (var, val) in enumerate(
        contrib.items(),
        start=1
    ):
        row[f'Top{i}_Variable'] = var
        row[f'Top{i}_Contribution'] = val

    rows.append(row)

top5_alarm_df = pd.DataFrame(rows)

# ==========================================================
# Display results
# ==========================================================
print(top5_alarm_df.head())

# Optional
# top5_alarm_df.to_excel(
#     'Top5_RBC_Contributors.xlsx',
#     index=False
# )

# ==========================================================
# Flare Event Detection
# ==========================================================
flare_threshold = 1.5                             # <-- set to your actual flare threshold
min_event_duration = pd.Timedelta(minutes=15)     # flare must stay above threshold this long to count
detection_window   = pd.Timedelta(minutes=15)     # look for a T2 exceedance within this long of event start

# ==========================================================
# Prepare flare series (sorted, clean index)
# ==========================================================
flare_df = Flare_data_clean_df[['Timestamp', 'Flare_Flowrate']].copy()
flare_df = flare_df.sort_values('Timestamp').reset_index(drop=True)

# ==========================================================
# Identify contiguous "flare above threshold" blocks
# ==========================================================
above = flare_df['Flare_Flowrate'] > flare_threshold

# assign a unique id to each contiguous run of True/False
block_id = (above != above.shift()).cumsum()

flare_events = []

for _, block in flare_df.groupby(block_id):

    if not above.loc[block.index[0]]:
        continue  # this block is "below threshold" — skip

    start_time = block['Timestamp'].iloc[0]
    end_time = block['Timestamp'].iloc[-1]
    duration = end_time - start_time

    if duration >= min_event_duration:
        flare_events.append({
            'Event_Start': start_time,
            'Event_End': end_time,
            'Duration_min': duration.total_seconds() / 60
        })

flare_events_df = pd.DataFrame(flare_events)

print(f"Total qualifying flare events (>= 15 min): {len(flare_events_df)}")

# ==========================================================
# Check whether T2 crossed its limit within 15 min of each event start
# (T2_limit here is a single scalar, since this is a global, not
# moving-window, PCA model)
# ==========================================================
pca_df = All_clean_df[['Timestamp', 'T2']].copy()
pca_df = pca_df.sort_values('Timestamp').reset_index(drop=True)

detected_flags = []
time_to_detect = []
first_hit_timestamps = []

for _, event in flare_events_df.iterrows():

    window_start = event['Event_Start']
    window_end = window_start + detection_window

    window_mask = (
        (pca_df['Timestamp'] >= window_start) &
        (pca_df['Timestamp'] < window_end)
    )

    window_data = pca_df.loc[window_mask]

    exceedances = window_data.loc[window_data['T2'] > T2_limit]

    detected = not exceedances.empty
    detected_flags.append(detected)

    if detected:
        first_hit = exceedances['Timestamp'].iloc[0]
        time_to_detect.append((first_hit - window_start).total_seconds() / 60)
        first_hit_timestamps.append(first_hit)
    else:
        time_to_detect.append(np.nan)
        first_hit_timestamps.append(pd.NaT)

flare_events_df['Detected'] = detected_flags
flare_events_df['Time_to_Detect_min'] = time_to_detect
flare_events_df['First_Hit_Timestamp'] = first_hit_timestamps

# ==========================================================
# Summary
# ==========================================================
n_events = len(flare_events_df)
n_detected = int(flare_events_df['Detected'].sum())
detection_rate = (n_detected / n_events * 100) if n_events > 0 else np.nan

print("\n" + "=" * 50)
print("Flare Event Detection Summary")
print("=" * 50)
print(f"Total flare events (>= 15 min) : {n_events}")
print(f"Detected by our approach       : {n_detected}")
print(f"Detection rate                 : {detection_rate:.1f}%")
print("=" * 50)

flare_events_df

Number of retained PCs = 98
T² limit = 160.20656012467592
Number of alarm samples: 41878
            Timestamp          T2     Top1_Variable  Top1_Contribution  \
0 2025-05-22 17:57:00  278.016863  99TI1321.DACA.PV           2.876621   
1 2025-05-22 17:58:00  277.299867  99TI1321.DACA.PV           2.791565   
2 2025-05-22 17:59:00  278.907159  99TI1321.DACA.PV           2.727872   
3 2025-05-22 18:00:00  278.656649  99TI1321.DACA.PV           2.649552   
4 2025-05-22 18:01:00  278.379471  99TI1321.DACA.PV           2.597831   

      Top2_Variable  Top2_Contribution     Top3_Variable  Top3_Contribution  \
0  99TI1320.DACA.PV           0.680746        PPC3401.PV           0.637696   
1  99TI4107.DACA.PV           0.647321  99TI1320.DACA.PV           0.637938   
2  99TI1320.DACA.PV           0.670790  99TI4107.DACA.PV           0.643859   
3  99TI4107.DACA.PV           0.656053  99TI1320.DACA.PV           0.634537   
4  99TI1320.DACA.PV           0.677827  99TI4107.DACA.PV           0.64

,Event_Start,Event_End,Duration_min,Detected,Time_to_Detect_min,First_Hit_Timestamp
0,2025-05-23 01:51:00,2025-05-23 02:07:00,16.0,True,0.0,2025-05-23 01:51:00
1,2025-05-23 09:04:00,2025-05-23 09:24:00,20.0,True,0.0,2025-05-23 09:04:00
2,2025-05-23 09:35:00,2025-05-23 11:26:00,111.0,True,0.0,2025-05-23 09:35:00
3,2025-05-23 11:30:00,2025-05-23 11:55:00,25.0,True,0.0,2025-05-23 11:30:00
4,2025-05-23 12:07:00,2025-05-23 14:26:00,139.0,True,0.0,2025-05-23 12:07:00
...,...,...,...,...,...,...
2289,2026-05-20 14:15:00,2026-05-20 14:41:00,26.0,False,NaN,NaT
2290,2026-05-20 16:06:00,2026-05-20 16:33:00,27.0,True,0.0,2026-05-20 16:06:00
2291,2026-05-20 16:41:00,2026-05-20 17:15:00,34.0,True,0.0,2026-05-20 16:41:00
2292,2026-05-20 18:07:00,2026-05-20 18:34:00,27.0,True,0.0,2026-05-20 18:07:00


In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import f

# ==========================================================
# Prepare data
# ==========================================================
X_df = All_clean_df.select_dtypes(include=np.number)

# Remove existing T2 column if present
X_df = X_df.drop(columns=['T2'], errors='ignore')

feature_names = X_df.columns
X = X_df.values

# ==========================================================
# Standardize
# ==========================================================
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# ==========================================================
# Global PCA model
# ==========================================================
pca = PCA()
scores = pca.fit_transform(Xs)

# Number of PCs to retain (95% variance)
cum_var = np.cumsum(
    pca.explained_variance_ratio_
)

A = np.argmax(cum_var >= 0.95) + 1

print(f'Number of retained PCs = {A}')

# ==========================================================
# Compute T²
# ==========================================================
Lambda = pca.explained_variance_[:A]
T = scores[:, :A]

T2 = np.sum(
    (T**2) / Lambda,
    axis=1
)

All_clean_df['T2'] = T2

# ==========================================================
# T² control limit
# ==========================================================
n = X.shape[0]
p = A

T2_limit = (1.5*
   ( p * (n - 1) * (n + 1)
    / (n * (n - p))
) * f.ppf(
    0.99,
    p,
    n - p
))

print('T² limit =', T2_limit)

# ==========================================================
# Reconstruction-Based Contribution (RBC)
# ==========================================================
P = pca.components_.T
P_A = P[:, :A]

# Reconstruction
X_rec = T @ P_A.T

# Residual matrix
E = Xs - X_rec

# RBC
RBC = E**2

RBC_df = pd.DataFrame(
    RBC,
    columns=feature_names,
    index=All_clean_df.index
)

# ==========================================================
# Alarm samples
# ==========================================================
alarm_idx = All_clean_df.loc[
    All_clean_df['T2'] > T2_limit
].index

print(
    'Number of alarm samples:',
    len(alarm_idx)
)

# ==========================================================
# Top-5 contributors for each alarm
# ==========================================================
rows = []

for idx in alarm_idx:

    contrib = (
        RBC_df.loc[idx]
        .sort_values(ascending=False)
        .head(5)
    )

    row = {
        'Timestamp':
            All_clean_df.loc[idx, 'Timestamp'],
        'T2':
            All_clean_df.loc[idx, 'T2']
    }

    for i, (var, val) in enumerate(
        contrib.items(),
        start=1
    ):
        row[f'Top{i}_Variable'] = var
        row[f'Top{i}_Contribution'] = val

    rows.append(row)

top5_alarm_df = pd.DataFrame(rows)

# ==========================================================
# Display results
# ==========================================================
print(top5_alarm_df.head())

# Optional
# top5_alarm_df.to_excel(
#     'Top5_RBC_Contributors.xlsx',
#     index=False
# )

# ==========================================================
# Flare Event Detection
# ==========================================================
flare_threshold = 1.5                             # <-- set to your actual flare threshold
min_event_duration = pd.Timedelta(minutes=15)     # flare must stay above threshold this long to count
detection_window   = pd.Timedelta(minutes=15)     # look for a T2 exceedance within this long of event start

# ==========================================================
# Prepare flare series (sorted, clean index)
# ==========================================================
flare_df = Flare_data_clean_df[['Timestamp', 'Flare_Flowrate']].copy()
flare_df = flare_df.sort_values('Timestamp').reset_index(drop=True)

# ==========================================================
# Identify contiguous "flare above threshold" blocks
# ==========================================================
above = flare_df['Flare_Flowrate'] > flare_threshold

# assign a unique id to each contiguous run of True/False
block_id = (above != above.shift()).cumsum()

flare_events = []

for _, block in flare_df.groupby(block_id):

    if not above.loc[block.index[0]]:
        continue  # this block is "below threshold" — skip

    start_time = block['Timestamp'].iloc[0]
    end_time = block['Timestamp'].iloc[-1]
    duration = end_time - start_time

    if duration >= min_event_duration:
        flare_events.append({
            'Event_Start': start_time,
            'Event_End': end_time,
            'Duration_min': duration.total_seconds() / 60
        })

flare_events_df = pd.DataFrame(flare_events)

print(f"Total qualifying flare events (>= 15 min): {len(flare_events_df)}")

# ==========================================================
# Check whether T2 crossed its limit within 15 min of each event start
# (T2_limit here is a single scalar, since this is a global, not
# moving-window, PCA model)
# ==========================================================
pca_df = All_clean_df[['Timestamp', 'T2']].copy()
pca_df = pca_df.sort_values('Timestamp').reset_index(drop=True)

detected_flags = []
time_to_detect = []
first_hit_timestamps = []

for _, event in flare_events_df.iterrows():

    window_start = event['Event_Start']
    window_end = window_start + detection_window

    window_mask = (
        (pca_df['Timestamp'] >= window_start) &
        (pca_df['Timestamp'] < window_end)
    )

    window_data = pca_df.loc[window_mask]

    exceedances = window_data.loc[window_data['T2'] > T2_limit]

    detected = not exceedances.empty
    detected_flags.append(detected)

    if detected:
        first_hit = exceedances['Timestamp'].iloc[0]
        time_to_detect.append((first_hit - window_start).total_seconds() / 60)
        first_hit_timestamps.append(first_hit)
    else:
        time_to_detect.append(np.nan)
        first_hit_timestamps.append(pd.NaT)

flare_events_df['Detected'] = detected_flags
flare_events_df['Time_to_Detect_min'] = time_to_detect
flare_events_df['First_Hit_Timestamp'] = first_hit_timestamps

# ==========================================================
# Summary
# ==========================================================
n_events = len(flare_events_df)
n_detected = int(flare_events_df['Detected'].sum())
detection_rate = (n_detected / n_events * 100) if n_events > 0 else np.nan

print("\n" + "=" * 50)
print("Flare Event Detection Summary")
print("=" * 50)
print(f"Total flare events (>= 15 min) : {n_events}")
print(f"Detected by our approach       : {n_detected}")
print(f"Detection rate                 : {detection_rate:.1f}%")
print("=" * 50)

flare_events_df

# ==========================================================
# RBC % contribution at the instant each event was detected
# (i.e., at First_Hit_Timestamp for rows where Detected == True)
# ==========================================================
top_n = 5
resample_rule = "5min"   # RBC resampling, kept consistent with earlier plots

RBC_plot = RBC_df.copy()
RBC_plot.index = All_clean_df["Timestamp"]

RBC_resampled = RBC_plot.resample(resample_rule).mean()

abs_contrib = RBC_resampled.abs()
row_totals = abs_contrib.sum(axis=1)
row_totals_safe = row_totals.replace(0, np.nan)

RBC_percent = abs_contrib.div(row_totals_safe, axis=0) * 100
RBC_percent = RBC_percent.fillna(0)

detected_events = flare_events_df[flare_events_df['Detected']].reset_index(drop=True)

rows = []

for _, event in detected_events.iterrows():

    instant = event['First_Hit_Timestamp']

    nearest_pos = RBC_percent.index.get_indexer([instant], method='nearest')[0]
    nearest_ts = RBC_percent.index[nearest_pos]

    contrib_at_instant = RBC_percent.iloc[nearest_pos]
    top_contributors = contrib_at_instant.sort_values(ascending=False).head(top_n)

    row = {
        "Event_Start": event['Event_Start'],
        "Event_End": event['Event_End'],
        "Duration_min": event['Duration_min'],
        "First_Hit_Timestamp": instant,
        "Matched_RBC_Timestamp": nearest_ts,
    }

    for rank, (var_name, pct) in enumerate(top_contributors.items(), start=1):
        row[f"Rank{rank}_Variable"] = var_name
        row[f"Rank{rank}_%Contribution"] = round(pct, 2)

    rows.append(row)

top_contributors_table = pd.DataFrame(rows)

print(f"\nDetected events with RBC % contribution table: {len(top_contributors_table)}")
top_contributors_table

Number of retained PCs = 98
T² limit = 200.25820015584495
Number of alarm samples: 27669
            Timestamp          T2     Top1_Variable  Top1_Contribution  \
0 2025-05-22 17:57:00  278.016863  99TI1321.DACA.PV           2.876621   
1 2025-05-22 17:58:00  277.299867  99TI1321.DACA.PV           2.791565   
2 2025-05-22 17:59:00  278.907159  99TI1321.DACA.PV           2.727872   
3 2025-05-22 18:00:00  278.656649  99TI1321.DACA.PV           2.649552   
4 2025-05-22 18:01:00  278.379471  99TI1321.DACA.PV           2.597831   

      Top2_Variable  Top2_Contribution     Top3_Variable  Top3_Contribution  \
0  99TI1320.DACA.PV           0.680746        PPC3401.PV           0.637696   
1  99TI4107.DACA.PV           0.647321  99TI1320.DACA.PV           0.637938   
2  99TI1320.DACA.PV           0.670790  99TI4107.DACA.PV           0.643859   
3  99TI4107.DACA.PV           0.656053  99TI1320.DACA.PV           0.634537   
4  99TI1320.DACA.PV           0.677827  99TI4107.DACA.PV           0.64

,Event_Start,Event_End,Duration_min,First_Hit_Timestamp,Matched_RBC_Timestamp,Rank1_Variable,Rank1_%Contribution,Rank2_Variable,Rank2_%Contribution,Rank3_Variable,Rank3_%Contribution,Rank4_Variable,Rank4_%Contribution,Rank5_Variable,Rank5_%Contribution
0,2025-05-23 01:51:00,2025-05-23 02:07:00,16.0,2025-05-23 01:51:00,2025-05-23 01:50:00,99TI1321.DACA.PV,20.58,99TI1320.DACA.PV,4.08,99TI4107.DACA.PV,3.79,99TI4105.DACA.PV,3.13,99TI3202A.DACA.PV,2.83
1,2025-05-23 09:04:00,2025-05-23 09:24:00,20.0,2025-05-23 09:04:00,2025-05-23 09:05:00,99TI1321.DACA.PV,24.10,99TI4107.DACA.PV,4.75,99TI1320.DACA.PV,3.66,99TI3202A.DACA.PV,3.62,42PC4208.PV,2.73
2,2025-05-23 09:35:00,2025-05-23 11:26:00,111.0,2025-05-23 09:35:00,2025-05-23 09:35:00,99TI1321.DACA.PV,18.79,700TI3219N,8.57,700TI3932N,8.08,99TI3202A.DACA.PV,3.46,PPI3206.PV,2.79
3,2025-05-23 11:30:00,2025-05-23 11:55:00,25.0,2025-05-23 11:30:00,2025-05-23 11:30:00,02TI219.PV,15.66,41TI4132.pv,14.70,99TI1321.DACA.PV,11.95,24TI3501.PV,5.17,99TI1515.DACA.PV,4.15
4,2025-05-23 12:07:00,2025-05-23 14:26:00,139.0,2025-05-23 12:07:00,2025-05-23 12:05:00,99TI1321.DACA.PV,23.95,99TI3202A.DACA.PV,4.18,99TI1320.DACA.PV,3.57,99TI1515.DACA.PV,2.74,99TI5406.DACA.PV,2.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,2026-05-19 21:27:00,2026-05-19 21:43:00,16.0,2026-05-19 21:37:00,2026-05-19 21:35:00,102PI4754R.DACA.PV,8.62,71TI8403.PV,6.51,71TC8702.PV,5.99,71TI0405.PV,3.91,103TC5604.daca.pv,3.57
173,2026-05-20 16:06:00,2026-05-20 16:33:00,27.0,2026-05-20 16:06:00,2026-05-20 16:05:00,71TI8403.PV,13.95,99TI2106.DACA.PV,10.55,71PC9801.PV,3.83,71TI1603.PV,2.99,99TI3202A.DACA.PV,2.67
174,2026-05-20 16:41:00,2026-05-20 17:15:00,34.0,2026-05-20 16:41:00,2026-05-20 16:40:00,71TI8403.PV,15.32,99TI2106.DACA.PV,7.05,71PC9801.PV,4.38,71TI1603.PV,3.19,103PC1101.daca.pv,2.87
175,2026-05-20 18:07:00,2026-05-20 18:34:00,27.0,2026-05-20 18:07:00,2026-05-20 18:05:00,71TI8403.PV,16.59,71TI1603.PV,6.02,508PC5103.DACA.PV,5.24,42PC4208.PV,4.77,71PC9801.PV,4.15
